In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

# Seed for reproducibility
np.random.seed(42)

# Define the time period (2 years, monthly data)
time_period = pd.date_range(start='2022-01-01', periods=24, freq='M')

# Generate a linear growth trend for customer acquisition with random fluctuations
base_growth = np.linspace(50, 150, num=len(time_period))  # Linear growth from 50 to 150 customers
random_fluctuation = np.random.normal(loc=0, scale=10, size=len(time_period))  # Random fluctuation around the base growth
monthly_customer_count = np.round(base_growth + random_fluctuation).astype(int)  # Total customers per month

# Ensure all customer counts are positive
monthly_customer_count[monthly_customer_count < 0] = 10

# Generate customer IDs and assign them to months
customer_ids = np.arange(1, sum(monthly_customer_count) + 1)
acquisition_months = np.repeat(time_period, monthly_customer_count)

# Generate customer segments with specified probabilities
segments = np.random.choice(['Young Professional', 'Mid-Career', 'Senior'], p=[0.15, 0.55, 0.30], size=len(customer_ids))

# Segment-based NPS adjustments
segment_nps_adjustments = {
    'Young Professional': 2.0,
    'Mid-Career': 1.5,
    'Senior': -1.5
}
nps_adjustments = np.array([segment_nps_adjustments[seg] for seg in segments])

# Simulate NPS scores influenced by segment
nps_scores = (np.random.normal(loc=5+nps_adjustments, scale=3, size=len(customer_ids))).astype(int)
nps_scores = np.clip(nps_scores, 0, 10)  # Ensure NPS scores are within valid range

# Lifespan calculation influenced by NPS score
lifespans = np.maximum((2 + 2 * nps_scores + np.random.normal(loc=0, scale=7, size=len(customer_ids))).astype(int), 0)

# Set consistent CAC per cohort based on initial monthly distribution
cac_base = np.random.normal(loc=50, scale=5, size=len(time_period))  # Base CAC values for each month
cac_costs = np.repeat(cac_base, monthly_customer_count)  # Repeat CAC value according to the number of new customers

# Monthly subscription fee remains constant
monthly_fee = 10.8

# Construct DataFrame
customers = pd.DataFrame({
    'Customer_ID': customer_ids,
    'Acquisition_Month': acquisition_months,
    'Segment': segments,
    'NPS_Score': nps_scores,
    'Expected_Lifespan_Months': lifespans,
    'CAC': cac_costs
})

# 1 incidencia en promedio por mes
technical_issues = np.random.poisson(lam=1, size=len(customers))

# Agregando al DataFrame
customers['Technical_Issues'] = technical_issues

# Calculate expected LTV for two years, capped at 24 months for simplicity
customers['Expected_LTV_in_Two_Years'] = (np.minimum(customers['Expected_Lifespan_Months'], 24) * monthly_fee).astype(int)

# Display the DataFrame
print(customers.head())

# Optionally, save the DataFrame to a CSV file for easier use in classes or further analysis
customers.to_csv('data/customer_data.csv', index=False)

   Customer_ID Acquisition_Month             Segment  NPS_Score  \
0            1        2022-01-31          Mid-Career          3   
1            2        2022-01-31  Young Professional          6   
2            3        2022-01-31          Mid-Career          5   
3            4        2022-01-31          Mid-Career          8   
4            5        2022-01-31  Young Professional         10   

   Expected_Lifespan_Months        CAC  Technical_Issues  \
0                         0  41.173062                 2   
1                         5  41.173062                 1   
2                        22  41.173062                 0   
3                        15  41.173062                 2   
4                        32  41.173062                 0   

   Expected_LTV_in_Two_Years  
0                          0  
1                         54  
2                        237  
3                        162  
4                        259  
